# 03. Debug Training and MMD Ablation

This notebook runs a short Replogle training job to validate the training path before launching longer experiments.

- Run 50 steps to check the dataloader, model, and losses.
- Set `optimization.MMD_loss_factor=0.0` for a minimal ablation that removes the distribution-level MMD loss.
- Treat this as a pipeline validation step rather than a reproduction of final benchmark numbers.

In [ ]:
from pathlib import Path
import shlex
import subprocess

# Set this to the PerturbDiff repository root.
# If the notebook server is launched from the repo root, the default is already correct.
PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "src" / "apps" / "run" / "rawdata_diffusion_sampling.py").exists(), \
    "Set PROJECT_ROOT to the PerturbDiff repository root before continuing."
DATASET_NAME = "replogle"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
PERTURB_ROOT = DATA_ROOT / "perturb_data"
RUN_DIR = PROJECT_ROOT / "runs" / "replogle_training_demo"
WANDB_DIR = PROJECT_ROOT / "wandb"

assert PERTURB_ROOT.exists(), "Run 00_download_data.ipynb first."
print("PERTURB_ROOT:", PERTURB_ROOT)
print("RUN_DIR:", RUN_DIR)

## 1. Define a Training Command Builder

The overrides can be read in four groups: data paths, model dimensions, cell-set settings, and training runtime settings.

In [ ]:
def build_train_cmd(run_name: str, max_steps: int = 50, mmd_loss_factor: float = 1.0):
    return [
        "python", "./src/apps/run/rawdata_diffusion_training.py",
        f"run_name={run_name}",
        "path=trixie_path",
        f"path.tmp_dir={PERTURB_ROOT}",
        f"path.diffusion.save_dir={RUN_DIR}",
        f"path.wandb.logging_dir={WANDB_DIR}",
        "data=replogle_finetune",
        "data.normalize_counts=10",
        "data.pad_length=2000",
        "data.embed_key=X_hvg",
        "model.hidden_num=[2000,512]",
        "model.input_dim=2000",
        "optimization.micro_batch_size=16",
        "data.use_cell_set=8",
        "optimization.optimizer.lr=0.002",
        f"optimization.MMD_loss_factor={mmd_loss_factor}",
        "cov_encoding=trixie_onehot",
        "cov_encoding.batch_encoding=onehot",
        "cov_encoding.celltype_encoding=llm",
        "cov_encoding.replogle_gene_encoding=genept",
        "model.p_drop_control=0",
        "data.keep_control_cell=false",
        "trainer.devices=[0]",
        f"trainer.max_steps={max_steps}",
        "trainer.limit_val_batches=1",
        "trainer.val_check_interval=25",
        "lightning.callbacks.checkpoint.save_top_k=-1",
        "lightning.callbacks.checkpoint.every_n_train_steps=50",
        "lightning.logger._target_=pytorch_lightning.loggers.logger.DummyLogger",
        "~lightning.logger.project",
        "~lightning.logger.save_dir",
        "~lightning.logger.name",
    ]

baseline_cmd = build_train_cmd("debug_replogle_mmd_on", max_steps=50, mmd_loss_factor=1.0)
print(" ".join(shlex.quote(x) for x in baseline_cmd))

## 2. Run Baseline Debug Training

The logs should include the diffusion denoising loss and the distribution-level MMD loss. The short 50-step run is intended to catch data, shape, and memory issues early.

In [ ]:
result = subprocess.run(baseline_cmd, cwd=PROJECT_ROOT, text=True)
print("return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Baseline training failed. Check the first error in the output above.")

## 3. Run the MMD-Off Ablation

This run changes only one parameter: `optimization.MMD_loss_factor=0.0`. Keeping all other settings fixed makes the ablation easier to interpret.

In [ ]:
ablation_cmd = build_train_cmd("debug_replogle_mmd_off", max_steps=50, mmd_loss_factor=0.0)
print(" ".join(shlex.quote(x) for x in ablation_cmd))

result = subprocess.run(ablation_cmd, cwd=PROJECT_ROOT, text=True)
print("return code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Ablation training failed. Check the first error in the output above.")

## 4. Inspect Checkpoints and Logs

The checkpoint produced here can be used in notebook 02 by replacing `model_checkpoint_path` with the generated `.ckpt` file.

In [ ]:
for pattern in ["*.ckpt", "*.log", "*.yaml"]:
    files = sorted(RUN_DIR.rglob(pattern))
    print(f"\n{pattern}")
    for path in files[:20]:
        size_mb = path.stat().st_size / 1024**2
        print(f"{size_mb:8.2f} MB  {path}")

## 5. Scale Up After the Debug Run

After the 50-step run succeeds, increase `max_steps` to 500 or more. If memory allows, increase `data.use_cell_set` from 8 to 16 or 32 to move closer to the official Replogle setting.